In [51]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import train_test_split
import scipy.stats as stat
from sklearn.preprocessing import LabelEncoder

Loading the Data and Selecting the Features

Import Data

In [76]:
loan_data_inputs_train = pd.read_csv('loan_data_inputs_train.csv', index_col = 0)
loan_data_targets_train = pd.read_csv('loan_data_targets_train.csv', index_col = 0) #, header = None)
loan_data_inputs_test = pd.read_csv('loan_data_inputs_test.csv', index_col = 0)
loan_data_targets_test = pd.read_csv('loan_data_targets_test.csv', index_col = 0)#, header = None)

In [53]:
# Convert boolean columns to integers (True=1, False=0)
loan_data_inputs_train = loan_data_inputs_train.astype({col: 'int64' for col in loan_data_inputs_train.select_dtypes('bool').columns})

In [54]:
#if 'addr_state:NM_VA' in loan_data_inputs_test.columns:
#    print("Column exists!")
loan_data_inputs_train.shape

(373028, 323)

In [55]:
loan_data_inputs_train.describe()

,Unnamed: 0.1,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,...,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>=86
count,373028.000000,3.730280e+05,3.730280e+05,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,3.730280e+05,373028.000000,...,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000
mean,233050.214038,1.308110e+07,1.459914e+07,14318.314778,14292.326514,14222.854516,13.827497,432.078820,7.322373e+04,17.221684,...,0.026263,0.256316,0.007726,0.865782,0.002911,0.004233,0.005359,0.060060,0.006718,0.056202
std,134586.411795,1.089460e+07,1.168336e+07,8287.668197,8275.446272,8298.860851,4.354152,243.523044,5.368944e+04,7.845668,...,0.159918,0.436599,0.087557,0.340887,0.053878,0.064923,0.073008,0.237598,0.081688,0.230312
min,0.000000,5.473400e+04,7.047300e+04,500.000000,500.000000,0.000000,5.420000,15.670000,1.896000e+03,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,116448.750000,3.639984e+06,4.379781e+06,8000.000000,8000.000000,8000.000000,10.990000,256.480000,4.500000e+04,11.370000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,233085.500000,1.010677e+07,1.194003e+07,12000.000000,12000.000000,12000.000000,13.660000,379.890000,6.289300e+04,16.880000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,349508.250000,2.073958e+07,2.300367e+07,20000.000000,20000.000000,19950.000000,16.490000,566.332500,8.880000e+04,22.780000,...,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,466283.000000,3.809811e+07,4.086083e+07,35000.000000,35000.000000,35000.000000,26.060000,1409.990000,7.141778e+06,39.990000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [56]:

# Convert columns containing numeric strings to float where possible
def convert_to_numeric(df):
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except ValueError:
            # Skip if conversion is not possible
            continue
convert_to_numeric(loan_data_inputs_train)

# Identify object columns (categorical)
object_columns = loan_data_inputs_train.select_dtypes('object').columns

# Step 1: Reduce high cardinality by grouping rare categories into "Other"
def reduce_cardinality(col, threshold=0.01):
    counts = col.value_counts(normalize=True)
    rare_labels = counts[counts < threshold].index
    return col.apply(lambda x: 'Other' if x in rare_labels else x)

for col in object_columns:
    if col in loan_data_inputs_train:
        loan_data_inputs_train[col] = reduce_cardinality(loan_data_inputs_train[col])

# Step 2: Use Label Encoding for all remaining categorical columns (no one-hot encoding)
label_encoders = {}
for col in object_columns:
    if col in loan_data_inputs_train.columns:
        le = LabelEncoder()
        loan_data_inputs_train[col] = le.fit_transform(loan_data_inputs_train[col].astype(str))
        label_encoders[col] = le  # Save encoders if needed for future use

# Convert any remaining columns to numeric if possible
loan_data_inputs_train = loan_data_inputs_train.apply(pd.to_numeric, errors='coerce')

# Fill NaN values with 0 or another strategy if needed
loan_data_inputs_train.fillna(0, inplace=True)

# Verify the number of columns (should be 324)
print(f"Number of columns after encoding: {loan_data_inputs_train.shape[1]}")

# Ensure that column names are kept the same
original_columns = loan_data_inputs_train.columns.tolist()[:324]  # Truncate if needed to keep only 324 columns
loan_data_inputs_train = loan_data_inputs_train[original_columns]

# Continue with logistic regression

Number of columns after encoding: 323


Explore Data

In [77]:
loan_data_targets_train.shape

(373028, 1)

In [78]:
loan_data_inputs_train.shape

(373028, 323)

In [79]:
loan_data_inputs_train.describe()

,Unnamed: 0.1,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,...,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>=86
count,373028.000000,3.730280e+05,3.730280e+05,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,3.730280e+05,373028.000000,...,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000,373028.000000
mean,233050.214038,1.308110e+07,1.459914e+07,14318.314778,14292.326514,14222.854516,13.827497,432.078820,7.322373e+04,17.221684,...,0.026263,0.256316,0.007726,0.865782,0.002911,0.004233,0.005359,0.060060,0.006718,0.056202
std,134586.411795,1.089460e+07,1.168336e+07,8287.668197,8275.446272,8298.860851,4.354152,243.523044,5.368944e+04,7.845668,...,0.159918,0.436599,0.087557,0.340887,0.053878,0.064923,0.073008,0.237598,0.081688,0.230312
min,0.000000,5.473400e+04,7.047300e+04,500.000000,500.000000,0.000000,5.420000,15.670000,1.896000e+03,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,116448.750000,3.639984e+06,4.379781e+06,8000.000000,8000.000000,8000.000000,10.990000,256.480000,4.500000e+04,11.370000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,233085.500000,1.010677e+07,1.194003e+07,12000.000000,12000.000000,12000.000000,13.660000,379.890000,6.289300e+04,16.880000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,349508.250000,2.073958e+07,2.300367e+07,20000.000000,20000.000000,19950.000000,16.490000,566.332500,8.880000e+04,22.780000,...,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,466283.000000,3.809811e+07,4.086083e+07,35000.000000,35000.000000,35000.000000,26.060000,1409.990000,7.141778e+06,39.990000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [80]:
loan_data_targets_test.shape

(93257, 1)

Selecting the Features

In [81]:
# Here we select a limited set of input variables in a new dataframe.
inputs_train_with_ref_cat = loan_data_inputs_train.loc[: , ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'home_ownership:OWN',
'home_ownership:MORTGAGE',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'addr_state:NM_VA',
'addr_state:NY',
'addr_state:OK_TN_MO_LA_MD_NC',
'addr_state:CA',
'addr_state:UT_KY_AZ_NJ',
'addr_state:AR_MI_PA_OH_MN',
'addr_state:RI_MA_DE_SD_IN',
'addr_state:GA_WA_OR',
'addr_state:WI_MT',
'addr_state:TX',
'addr_state:IL_CT',
'addr_state:KS_SC_CO_VT_AK_MS',
'addr_state:WV_NH_WY_DC_ME_ID',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:oth__med__vacation',
'purpose:major_purch__car__home_impr',
'initial_list_status:f',
'initial_list_status:w',
'term:36',
'term:60',
'emp_length:0',
'emp_length:1',
'emp_length:2-4',
'emp_length:5-6',
'emp_length:7-9',
'emp_length:10',
'mths_since_issue_d:<38',
'mths_since_issue_d:38-39',
'mths_since_issue_d:40-41',
'mths_since_issue_d:42-48',
'mths_since_issue_d:49-52',
'mths_since_issue_d:53-64',
'mths_since_issue_d:65-84',
'mths_since_issue_d:>84',
'int_rate:<9.548',
'int_rate:9.548-12.025',
'int_rate:12.025-15.74',
'int_rate:15.74-20.281',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'mths_since_earliest_cr_line:141-164',
'mths_since_earliest_cr_line:165-247',
'mths_since_earliest_cr_line:248-270',
'mths_since_earliest_cr_line:271-352',
'mths_since_earliest_cr_line:>352',
'delinq_2yrs:0',
'delinq_2yrs:1-3',
'delinq_2yrs:>=4',
'inq_last_6mths:0',
'inq_last_6mths:1-2',
'inq_last_6mths:3-6',
'inq_last_6mths:>6',
'open_acc:0',
'open_acc:1-3',
'open_acc:4-12',
'open_acc:13-17',
'open_acc:18-22',
'open_acc:23-25',
'open_acc:26-30',
'open_acc:>=31',
'pub_rec:0-2',
'pub_rec:3-4',
'pub_rec:>=5',
'total_acc:<=27',
'total_acc:28-51',
'total_acc:>=52',
'acc_now_delinq:0',
'acc_now_delinq:>=1',
'total_rev_hi_lim:<=5K',
'total_rev_hi_lim:5K-10K',
'total_rev_hi_lim:10K-20K',
'total_rev_hi_lim:20K-30K',
'total_rev_hi_lim:30K-40K',
'total_rev_hi_lim:40K-55K',
'total_rev_hi_lim:55K-95K',
'total_rev_hi_lim:>95K',
'annual_inc:<20K',
'annual_inc:20K-30K',
'annual_inc:30K-40K',
'annual_inc:40K-50K',
'annual_inc:50K-60K',
'annual_inc:60K-70K',
'annual_inc:70K-80K',
'annual_inc:80K-90K',
'annual_inc:90K-100K',
'annual_inc:100K-120K',
'annual_inc:120K-140K',
'annual_inc:>140K',
'dti:<=1.4',
'dti:1.4-3.5',
'dti:3.5-7.7',
'dti:7.7-10.5',
'dti:10.5-16.1',
'dti:16.1-20.3',
'dti:20.3-21.7',
'dti:21.7-22.4',
'dti:22.4-35',
'dti:>35',
'mths_since_last_delinq:Missing',
'mths_since_last_delinq:0-3',
'mths_since_last_delinq:4-30',
'mths_since_last_delinq:31-56',
'mths_since_last_delinq:>=57',
'mths_since_last_record:Missing',
'mths_since_last_record:0-2',
'mths_since_last_record:3-20',
'mths_since_last_record:21-31',
'mths_since_last_record:32-80',
'mths_since_last_record:81-86',
'mths_since_last_record:>=86'
]]




In [82]:
loan_data_inputs_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 373028 entries, 427211 to 121958
Columns: 323 entries, Unnamed: 0.1 to mths_since_last_record:>=86
dtypes: float64(49), int64(242), object(32)
memory usage: 922.1+ MB


In [83]:
# Here we store the names of the reference category dummy variables in a list.
ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'initial_list_status:f',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'delinq_2yrs:>=4',
'inq_last_6mths:>6',
'open_acc:0',
'pub_rec:0-2',
'total_acc:<=27',
'acc_now_delinq:0',
'total_rev_hi_lim:<=5K',
'annual_inc:<20K',
'dti:>35',
'mths_since_last_delinq:0-3',
'mths_since_last_record:0-2']

In [84]:
inputs_train = inputs_train_with_ref_cat.drop(ref_categories, axis = 1)
# From the dataframe with input variables, we drop the variables with variable names in the list with reference categories. 
inputs_train.head()

,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:OWN,home_ownership:MORTGAGE,addr_state:NM_VA,addr_state:NY,...,mths_since_last_delinq:Missing,mths_since_last_delinq:4-30,mths_since_last_delinq:31-56,mths_since_last_delinq:>=57,mths_since_last_record:Missing,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>=86
427211,1,0,0,0,0,0,0,1,0,0,...,1,0,0,0,1,0,0,0,0,0
206088,0,0,1,0,0,0,0,1,0,0,...,0,1,0,0,1,0,0,0,0,0
136020,1,0,0,0,0,0,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
412305,0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
36159,0,0,1,0,0,0,0,1,0,0,...,1,0,0,0,1,0,0,0,0,0


PD Model Estimation

Logistic Regression

In [85]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import numpy as np

In [86]:
reg = LogisticRegression()
# We create an instance of an object from the 'LogisticRegression' class.

In [87]:
pd.options.display.max_rows = None
# Sets the pandas dataframe options to display all columns/ rows.

In [88]:
loan_data_targets_train.shape

(373028, 1)

In [89]:
loan_data_targets_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 373028 entries, 427211 to 121958
Data columns (total 1 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   good_bad  373028 non-null  int64
dtypes: int64(1)
memory usage: 5.7 MB


In [90]:
inputs_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 373028 entries, 427211 to 121958
Columns: 104 entries, grade:A to mths_since_last_record:>=86
dtypes: int64(104)
memory usage: 298.8 MB


In [91]:
type(loan_data_targets_train)

pandas.core.frame.DataFrame

In [92]:
# Get all object type columns
object_columns = loan_data_inputs_train.select_dtypes(include='object').columns

# Get all numeric type columns
numeric_columns = loan_data_inputs_train.select_dtypes(include=['int64', 'float64']).columns


In [93]:
print(object_columns)

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'issue_d', 'loan_status',
       'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code',
       'addr_state', 'earliest_cr_line', 'initial_list_status', 'last_pymnt_d',
       'next_pymnt_d', 'last_credit_pull_d', 'application_type',
       'earliest_cr_line_date', 'issue_d_date', 'mths_since_issue_d_factor',
       'int_rate_factor', 'mths_since_earliest_cr_line_factor',
       'total_acc_factor', 'total_rev_hi_lim_factor', 'installment_factor',
       'annual_inc_factor', 'dti_factor'],
      dtype='object')


In [94]:
print(numeric_columns)

Index(['Unnamed: 0.1', 'id', 'member_id', 'loan_amnt', 'funded_amnt',
       'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti',
       ...
       'dti:21.7-22.4', 'dti:22.4-35', 'dti:>35',
       'mths_since_last_record:Missing', 'mths_since_last_record:0-2',
       'mths_since_last_record:3-20', 'mths_since_last_record:21-31',
       'mths_since_last_record:32-80', 'mths_since_last_record:81-86',
       'mths_since_last_record:>=86'],
      dtype='object', length=291)


In [95]:

reg.fit(inputs_train, loan_data_targets_train)
# Estimates the coefficients of the object from the 'LogisticRegression' class
# with inputs (independent variables) contained in the first dataframe
# and targets (dependent variables) contained in the second dataframe.

/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [97]:
LogisticRegression()
#In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.
#On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.

LogisticRegression()

In [98]:
reg.intercept_
# Displays the intercept contain in the estimated ("fitted") object from the 'LogisticRegression' class.

# array([-1.63136862])

array([-1.28874267])

In [99]:
reg.coef_
# Displays the coefficients contained in the estimated ("fitted") object from the 'LogisticRegression' class.

array([[ 1.15663196,  0.90502001,  0.70632005,  0.51578342,  0.33282703,
         0.14080988,  0.09274447,  0.10683968,  0.0326291 ,  0.05654255,
         0.05703365,  0.06017564,  0.07639445,  0.13363006,  0.09715783,
         0.18257796,  0.23487662,  0.2215337 ,  0.26066601,  0.3178653 ,
         0.51721675,  0.08719059, -0.01021118,  0.30625607,  0.20253302,
         0.21507808,  0.26896546,  0.05307856,  0.07681095,  0.10060589,
         0.12667899,  0.0920508 ,  0.06133547,  0.12436498,  1.08432768,
         0.88205748,  0.78241405,  0.5790092 ,  0.41732443,  0.17128462,
        -0.06565546,  0.85395701,  0.53873511,  0.2896894 ,  0.10332835,
         0.05800299,  0.0410265 ,  0.08271036,  0.1236028 ,  0.13061037,
         0.0713404 ,  0.03473491,  0.66117516,  0.51882424,  0.30905225,
        -0.00184972, -0.10159177, -0.1269035 , -0.142367  , -0.14566326,
        -0.10858332, -0.19955764,  0.12039839,  0.17340197, -0.02074232,
         0.02272428,  0.22975503,  0.03757   ,  0.0

In [100]:
feature_name = inputs_train.columns.values
# Stores the names of the columns of a dataframe in a variable.

In [101]:
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
# Creates a dataframe with a column titled 'Feature name' and row values contained in the 'feature_name' variable.
summary_table['Coefficients'] = np.transpose(reg.coef_)
# Creates a new column in the dataframe, called 'Coefficients',
# with row values the transposed coefficients from the 'LogisticRegression' object.
summary_table.index = summary_table.index + 1
# Increases the index of every row of the dataframe with 1.
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
# Assigns values of the row with index 0 of the dataframe.
summary_table = summary_table.sort_index()
# Sorts the dataframe by index.
summary_table

,Feature name,Coefficients
0,Intercept,-1.288743
1,grade:A,1.156632
2,grade:B,0.905020
3,grade:C,0.706320
4,grade:D,0.515783
5,grade:E,0.332827
6,grade:F,0.140810
7,home_ownership:OWN,0.092744
8,home_ownership:MORTGAGE,0.106840
9,addr_state:NM_VA,0.032629


Build a Logistic Regression Model with P-Values

In [102]:
# P values for sklearn logistic regression.

# Class to display p-values for logistic regression in sklearn.

from sklearn import linear_model
import scipy.stats as stat

class LogisticRegression_with_p_values:
    
    def __init__(self,*args,**kwargs):#,**kwargs):
        self.model = linear_model.LogisticRegression(*args,**kwargs)#,**args)

    def fit(self,X,y):
        self.model.fit(X,y)
        
        #### Get p-values for the fitted model ####
        denom = (2.0 * (1.0 + np.cosh(self.model.decision_function(X))))
        denom = np.tile(denom,(X.shape[1],1)).T
        F_ij = np.dot((X / denom).T,X) ## Fisher Information Matrix
        Cramer_Rao = np.linalg.inv(F_ij) ## Inverse Information Matrix
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates # z-score for eaach model coefficient
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores] ### two tailed test for p-values
        
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values

In [103]:
reg = LogisticRegression_with_p_values()
# We create an instance of an object from the newly created 'LogisticRegression_with_p_values()' class.

In [104]:
# Convert target data to 1D array
#loan_data_targets_train = loan_data_targets_train.values.ravel()

#np.isnan(loan_data_targets_train).sum()
#type(loan_data_targets_train)

In [105]:
loan_data_inputs_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 373028 entries, 427211 to 121958
Columns: 323 entries, Unnamed: 0.1 to mths_since_last_record:>=86
dtypes: float64(49), int64(242), object(32)
memory usage: 922.1+ MB


In [106]:
loan_data_targets_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 373028 entries, 427211 to 121958
Data columns (total 1 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   good_bad  373028 non-null  int64
dtypes: int64(1)
memory usage: 5.7 MB


In [107]:
reg.fit(inputs_train, loan_data_targets_train)
# Estimates the coefficients of the object from the 'LogisticRegression' class
# with inputs (independent variables) contained in the first dataframe
# and targets (dependent variables) contained in the second dataframe.

/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [108]:
# Same as above.
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table

,Feature name,Coefficients
0,Intercept,-1.288743
1,grade:A,1.156632
2,grade:B,0.905020
3,grade:C,0.706320
4,grade:D,0.515783
5,grade:E,0.332827
6,grade:F,0.140810
7,home_ownership:OWN,0.092744
8,home_ownership:MORTGAGE,0.106840
9,addr_state:NM_VA,0.032629


In [109]:
# This is a list.
p_values = reg.p_values
# We take the result of the newly added method 'p_values' and store it in a variable 'p_values'.

In [110]:
# Add the intercept for completeness.
p_values = np.append(np.nan, np.array(p_values))
# We add the value 'NaN' in the beginning of the variable with p-values.

In [111]:
summary_table['p_values'] = p_values
# In the 'summary_table' dataframe, we add a new column, called 'p_values', containing the values from the 'p_values' variable.

In [112]:
summary_table

,Feature name,Coefficients,p_values
0,Intercept,-1.288743,NaN
1,grade:A,1.156632,3.224871e-37
2,grade:B,0.905020,1.324943e-49
3,grade:C,0.706320,1.199528e-35
4,grade:D,0.515783,9.366579e-22
5,grade:E,0.332827,4.666413e-12
6,grade:F,0.140810,5.009530e-03
7,home_ownership:OWN,0.092744,4.370196e-06
8,home_ownership:MORTGAGE,0.106840,4.122738e-17
9,addr_state:NM_VA,0.032629,3.089441e-01


In [113]:
# We are going to remove some features, the coefficients for all or almost all of the dummy variables for which,
# are not statistically significant.

# We do that by specifying another list of dummy variables as reference categories, and a list of variables to remove.
# Then, we are going to drop the two datasets from the original list of dummy variables.

# Conventionally, if a p-value is lower than 0.05, we conclude that the coefficient of a variable is statistically significant

# Variables
inputs_train_with_ref_cat = loan_data_inputs_train.loc[: , ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'home_ownership:OWN',
'home_ownership:MORTGAGE',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'addr_state:NM_VA',
'addr_state:NY',
'addr_state:OK_TN_MO_LA_MD_NC',
'addr_state:CA',
'addr_state:UT_KY_AZ_NJ',
'addr_state:AR_MI_PA_OH_MN',
'addr_state:RI_MA_DE_SD_IN',
'addr_state:GA_WA_OR',
'addr_state:WI_MT',
'addr_state:TX',
'addr_state:IL_CT',
'addr_state:KS_SC_CO_VT_AK_MS',
'addr_state:WV_NH_WY_DC_ME_ID',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:oth__med__vacation',
'purpose:major_purch__car__home_impr',
'initial_list_status:f',
'initial_list_status:w',
'term:36',
'term:60',
'emp_length:0',
'emp_length:1',
'emp_length:2-4',
'emp_length:5-6',
'emp_length:7-9',
'emp_length:10',
'mths_since_issue_d:<38',
'mths_since_issue_d:38-39',
'mths_since_issue_d:40-41',
'mths_since_issue_d:42-48',
'mths_since_issue_d:49-52',
'mths_since_issue_d:53-64',
'mths_since_issue_d:65-84',
'mths_since_issue_d:>84',
'int_rate:<9.548',
'int_rate:9.548-12.025',
'int_rate:12.025-15.74',
'int_rate:15.74-20.281',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'mths_since_earliest_cr_line:141-164',
'mths_since_earliest_cr_line:165-247',
'mths_since_earliest_cr_line:248-270',
'mths_since_earliest_cr_line:271-352',
'mths_since_earliest_cr_line:>352',
'inq_last_6mths:0',
'inq_last_6mths:1-2',
'inq_last_6mths:3-6',
'inq_last_6mths:>6',
'acc_now_delinq:0',
'acc_now_delinq:>=1',
'annual_inc:<20K',
'annual_inc:20K-30K',
'annual_inc:30K-40K',
'annual_inc:40K-50K',
'annual_inc:50K-60K',
'annual_inc:60K-70K',
'annual_inc:70K-80K',
'annual_inc:80K-90K',
'annual_inc:90K-100K',
'annual_inc:100K-120K',
'annual_inc:120K-140K',
'annual_inc:>140K',
'dti:<=1.4',
'dti:1.4-3.5',
'dti:3.5-7.7',
'dti:7.7-10.5',
'dti:10.5-16.1',
'dti:16.1-20.3',
'dti:20.3-21.7',
'dti:21.7-22.4',
'dti:22.4-35',
'dti:>35',
'mths_since_last_delinq:Missing',
'mths_since_last_delinq:0-3',
'mths_since_last_delinq:4-30',
'mths_since_last_delinq:31-56',
'mths_since_last_delinq:>=57',
'mths_since_last_record:Missing',
'mths_since_last_record:0-2',
'mths_since_last_record:3-20',
'mths_since_last_record:21-31',
'mths_since_last_record:32-80',
'mths_since_last_record:81-86',
'mths_since_last_record:>=86',
]]

In [114]:
ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'initial_list_status:f',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'inq_last_6mths:>6',
'acc_now_delinq:0',
'annual_inc:<20K',
'dti:>35',
'mths_since_last_delinq:0-3',
'mths_since_last_record:0-2']

In [115]:
inputs_train = inputs_train_with_ref_cat.drop(ref_categories, axis = 1)
inputs_train.head()

,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:OWN,home_ownership:MORTGAGE,addr_state:NM_VA,addr_state:NY,...,mths_since_last_delinq:Missing,mths_since_last_delinq:4-30,mths_since_last_delinq:31-56,mths_since_last_delinq:>=57,mths_since_last_record:Missing,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>=86
427211,1,0,0,0,0,0,0,1,0,0,...,1,0,0,0,1,0,0,0,0,0
206088,0,0,1,0,0,0,0,1,0,0,...,0,1,0,0,1,0,0,0,0,0
136020,1,0,0,0,0,0,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
412305,0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
36159,0,0,1,0,0,0,0,1,0,0,...,1,0,0,0,1,0,0,0,0,0


In [116]:
# Here we run a new model.
reg2 = LogisticRegression_with_p_values()
reg2.fit(inputs_train, loan_data_targets_train)

/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
/Users/paninikalwankar/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [117]:
feature_name = inputs_train.columns.values

In [118]:
# Same as above.
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg2.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg2.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table

,Feature name,Coefficients
0,Intercept,-1.285478
1,grade:A,1.141220
2,grade:B,0.885134
3,grade:C,0.690067
4,grade:D,0.502034
5,grade:E,0.326603
6,grade:F,0.136833
7,home_ownership:OWN,0.091909
8,home_ownership:MORTGAGE,0.106068
9,addr_state:NM_VA,0.028893


In [119]:
# We add the 'p_values' here, just as we did before.
p_values = reg2.p_values
p_values = np.append(np.nan,np.array(p_values))
summary_table['p_values'] = p_values
summary_table
# Here we get the results for our final PD model.

,Feature name,Coefficients,p_values
0,Intercept,-1.285478,NaN
1,grade:A,1.141220,1.832370e-36
2,grade:B,0.885134,7.377043e-48
3,grade:C,0.690067,1.779183e-34
4,grade:D,0.502034,4.885332e-21
5,grade:E,0.326603,4.525712e-12
6,grade:F,0.136833,5.268102e-03
7,home_ownership:OWN,0.091909,5.027123e-06
8,home_ownership:MORTGAGE,0.106068,4.970273e-17
9,addr_state:NM_VA,0.028893,3.668085e-01


In [120]:
import pickle

In [121]:
pickle.dump(reg2, open('pd_model.sav', 'wb'))
# Here we export our model to a 'SAV' file with file name 'pd_model.sav'.